In [ ]:
from systemflow.hep.utils import hep_graph_from_spreadsheet, hep_with_updated_parameters, hep_construct_graph, dataframes_from_spreadsheet
from systemflow.hep.metrics import Productivity
from systemflow.classifier import L1TClassifier, get_passed
from systemflow.metrics import precision, recall, f1_score
from systemflow.models import density_scale_model
from copy import deepcopy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import pickle as pkl

from scipy.optimize import curve_fit, approx_fprime
from multiprocess import Pool, cpu_count

In [ ]:
n_cpus = cpu_count()

In [ ]:
# load data from the spreadsheet which defines the structure of the workflow,
# as well as the parameters for data rates, efficiency, data reduction, and classifier performance
# ...these are taken from predictions for the Run-5 CMS
run5_det, run5_proc, run5_globals = dataframes_from_spreadsheet("configurations/cms_system_200.xlsx")
run5_smartpx_det, run5_smartpx_proc, run5_smartpx_globals = dataframes_from_spreadsheet("configurations/cms_system_200_smartpx.xlsx")

In [ ]:
run5_det

In [ ]:
run5_proc

In [ ]:
#import the data predicting wall time scaling by pileup
scaling = pd.read_excel("wall time scaling.xlsx", sheet_name="Data")
#fit a polynomial to this data for CPU and GPU runtimes
fit_poly = lambda x, k3, k2, k1: k3 * x ** 3 + k2 * x ** 2 + k1 * x
k, cv = curve_fit(fit_poly, scaling["Size"], scaling["Wall Time"])
k_gpu, cv_gpu = curve_fit(fit_poly, scaling["Size"], scaling["Wall Time GPU"])

In [ ]:
#define a dictionary with functions defining the scaling of trigger runtimes with incoming data
funcs = {"Global": lambda x: fit_poly(x, *k), "Intermediate": lambda x: x / 2.0e6}
funcs_gpu = {"Global": lambda x: fit_poly(x, *k_gpu), "Intermediate": lambda x: x / 2.0e6}

In [ ]:
"""
Build base graph definitions from the spreadsheet configurations.
In v2.0, graphs are ExecutionGraph objects constructed from spreadsheets.
Calling graph_def() executes the graph and returns a result with metric_values.
"""
ex_base = hep_graph_from_spreadsheet("configurations/cms_system_200.xlsx", functions=funcs)
ex_gpu = hep_graph_from_spreadsheet("configurations/cms_system_200.xlsx", functions=funcs_gpu)
ex_reduction = hep_graph_from_spreadsheet("configurations/cms_system_200_smartpx.xlsx", functions=funcs_gpu)

In [ ]:
#current l1t accept / skill - set L1T reduction ratio to 400
ex = hep_with_updated_parameters(ex_base, {"Intermediate": {"reduction ratio (1)": 400}})

In [ ]:
ex.get_node("Intermediate").parameters["reduction ratio (1)"]

In [ ]:
ex.get_node("Intermediate").parameters["classifier (obj)"].skill_boost

In [ ]:
# Execute the graph to inspect power
ex_result = ex()
ex_result.metric_values["total power (W)"] / 1e6

In [ ]:
ex_result.get_node("Intermediate").properties["output rate (Hz)"]

In [ ]:
ex_result.get_node("Intermediate").properties["input rate (Hz)"] / ex_result.get_node("Intermediate").properties["output rate (Hz)"]

In [ ]:
ex_result.get_node("Intermediate").properties["input rate (Hz)"]

In [ ]:
ex_result.get_node("Global").properties["input rate (Hz)"] / ex_result.get_node("Global").properties["output rate (Hz)"]

In [ ]:
ex_result.get_node("Global").properties["output rate (Hz)"]

In [ ]:
ex_result.get_node("Global").properties["input rate (Hz)"]

In [ ]:
def extract_results(graph):
    """Extract power and confusion matrix from an executed v2.0 graph."""
    mv = graph.metric_values
    power = mv["total power (W)"]
    confusion = mv["pipeline contingency (2x2)"]
    return power, confusion

In [ ]:
def vary_system(base_def, reduction_ratio, skill):
    """Create a variant of the system with updated L1T parameters, execute, and return results."""
    classifier = deepcopy(base_def.get_node("Intermediate").parameters["classifier (obj)"])
    classifier.skill_boost = skill
    variant = hep_with_updated_parameters(base_def, {
        "Intermediate": {"reduction ratio (1)": reduction_ratio, "classifier (obj)": classifier}
    })
    result = variant()
    mv = result.metric_values
    power = mv["total power (W)"]
    confusion = mv["pipeline contingency (2x2)"]
    return power, confusion

In [ ]:
baseline = vary_system(ex, 400, 0.0)

In [ ]:
tracking_l1t = vary_system(ex, 400, 0.4)

In [ ]:
gpu_system = vary_system(ex_gpu, 53.3, 0.0)
phase2_system = vary_system(ex_gpu, 53.3, 0.4)

In [ ]:
smart_system = vary_system(ex_reduction, 53.3, 0.4)

In [ ]:
system_labels = ["Phase-1", "L1 Tracks", "GPU", "L1 Tracks + GPU", "L1T Tracks, GPU, Smart Sensing"]

In [ ]:
all_systems = [baseline, tracking_l1t, gpu_system, phase2_system, smart_system]

In [ ]:
r = vary_system(ex, 100, 0.0)

In [ ]:
r[0] / 1e6

In [ ]:
#its predicted confusion matrix for the workflow
r[1]

In [ ]:
#vary this accept rate from today's rate to the planned Run-5 
l1t_reductions = np.linspace(450, 40, 101)
l1t_skills = np.linspace(0, 1.0, 101)

In [ ]:
pmap_args = [(ex, r, s) for r in l1t_reductions for s in l1t_skills]

In [ ]:
def map_fn(args):
    return vary_system(*args)

In [ ]:
with Pool(n_cpus) as pool:
    res = pool.map(map_fn, pmap_args)

In [ ]:
with open("result.pkl", "wb") as f:
    pkl.dump(res, f)

In [ ]:
# with open("result.pkl", "rb") as f:
#     res = pkl.load(f)

In [ ]:
len(res)

In [ ]:
range(0, 101**2, 101)

In [ ]:
res2 = [res[i:i+len(l1t_skills)] for i in range(0, len(l1t_skills)*len(l1t_reductions), len(l1t_skills))]

In [ ]:
len(res2)

In [ ]:
len(res2[0])

In [ ]:
def sys_productivity(confusion, power):
    n = np.sum(get_passed(confusion))
    f1 = f1_score(confusion)
    productivity = (n * f1) / power
    return productivity

In [ ]:
def extract_metrics(results):
    all_confusion = np.array([r[1] for r in results])

    all_power = [r[0] / density_scale_model(2032) for r in results]
    all_power = np.array(all_power)

    all_recall = np.array([recall(all_confusion[i,:,:]) for i in range(all_confusion.shape[0])])
    all_precision = np.array([precision(all_confusion[i,:,:]) for i in range(all_confusion.shape[0])])
    all_f1 = np.array([f1_score(all_confusion[i,:,:]) for i in range(all_confusion.shape[0])])
    all_productivity = [sys_productivity(all_confusion[i,:,:], all_power[i]) for i in range(all_confusion.shape[0])]

    metrics = {"confusion": all_confusion,
               "power": all_power,
               "recall": all_recall,
               "precision": all_precision,
               "f1 score": all_f1,
               "productivity": all_productivity,}

    return metrics

In [ ]:
run5_metrics = [extract_metrics(r) for r in res2]

In [ ]:
res_f1 = np.stack([r["f1 score"] for r in run5_metrics]).transpose()

In [ ]:
res_recall = np.stack([r["recall"] for r in run5_metrics]).transpose()

In [ ]:
res_precision = np.stack([r["precision"] for r in run5_metrics]).transpose()

In [ ]:
power = np.stack([r["power"] for r in run5_metrics]).transpose()

In [ ]:
res_productivity = np.stack([r["productivity"] for r in run5_metrics]).transpose()

In [ ]:
from scipy.ndimage import gaussian_filter

In [ ]:
smoothed_f1 = gaussian_filter(res_f1, sigma=3)

In [ ]:
smoothed_productivity = gaussian_filter(res_productivity, sigma=3)

In [ ]:
np.savez_compressed("smoothed_f1.npz", smoothed_f1)

In [ ]:
c = ex.get_node("Intermediate").parameters["classifier (obj)"]

In [ ]:
fig = go.Figure(data = go.Histogram(x = c.null_scores, name = "False"))
fig.add_trace(go.Histogram(x = c.pos_scores, name = "True"))
fig.update_layout(width =800, height = 600,
                  title = "Histogram of L1T Classifier Model Scores")
fig.show()

In [ ]:
systems_f1 = np.array([f1_score(s[1]) for s in all_systems])

In [ ]:
systems_power = np.array([s[0] / density_scale_model(2032) for s in all_systems])

In [ ]:
systems_power / 1e6

In [ ]:
systems_reductions = np.array([400, 400, 53.3, 53.3, 53.3])
l1t_improvement = np.array([0.0, 0.4, 0.0, 0.4, 0.4])

In [ ]:
fig = go.Figure(data =
    go.Contour(
        z=smoothed_f1,
        x=l1t_reductions, # horizontal axis
        y=l1t_skills, # vertical axis,
        contours = dict(showlabels = True),
        colorbar = dict(title = "F1 Score")
         
    ),
    )

y_offset = 0.015
fig.add_trace(go.Scatter(x = systems_reductions[0:1],
                        y = l1t_improvement[0:1] + y_offset,
                        mode = "markers",
                        marker = dict(size = 14, color = "gray", symbol="circle"),
                        name = "Phase-1"))

fig.add_trace(go.Scatter(x = systems_reductions[1:2],
                        y = l1t_improvement[1:2],
                        mode = "markers",
                        marker = dict(size = 14, color = "gray", symbol = "square"),
                        name = "L1 Tracks"))

fig.add_trace(go.Scatter(x = systems_reductions[2:3],
                        y = l1t_improvement[2:3] + y_offset,
                        mode = "markers",
                        marker = dict(size = 14, color = "gray", symbol = "cross"),
                        name = "Increased L1T Accept"))

fig.add_trace(go.Scatter(x = systems_reductions[3:4],
                        y = l1t_improvement[3:4],
                        mode = "markers",
                        marker = dict(size = 14, color = "gray", symbol = "star"),
                        name = "Phase-2"))

fig.update_layout(width = 800, 
                  height = 600,
                  xaxis_title = "L1T Reduction Ratio",
                  yaxis_title = "L1T Skill Improvement",
                  title = "F1 Score by L1T Skill & Reduction Ratio",
                  legend=dict(xanchor = "right",
                    x = 0.95))
fig.update_xaxes(autorange="reversed")
fig.update_yaxes(range=[0.0, 0.8])
fig.show()

In [ ]:
fig = go.Figure(data =
    go.Contour(
        z=smoothed_f1,
        x=l1t_reductions, # horizontal axis
        y=l1t_skills, # vertical axis,
        contours = dict(showlabels = True),
        colorbar = dict(title = "F1 Score")
         
    ),
    )

y_offset = 0.015
fig.add_trace(go.Scatter(x = systems_reductions[0:1],
                        y = l1t_improvement[0:1] + y_offset,
                        mode = "markers",
                        marker = dict(size = 14, color = "gray", symbol="circle"),
                        name = "Phase-1"))

fig.add_trace(go.Scatter(x = systems_reductions[1:2],
                        y = l1t_improvement[1:2],
                        mode = "markers",
                        marker = dict(size = 14, color = "gray", symbol = "square"),
                        name = "L1 Tracks"))

fig.add_trace(go.Scatter(x = systems_reductions[2:3],
                        y = l1t_improvement[2:3] + y_offset,
                        mode = "markers",
                        marker = dict(size = 14, color = "gray", symbol = "cross"),
                        name = "Increased L1T Accept"))

fig.add_trace(go.Scatter(x = systems_reductions[3:4],
                        y = l1t_improvement[3:4],
                        mode = "markers",
                        marker = dict(size = 14, color = "gray", symbol = "star"),
                        name = "Phase-2"))

fig.update_layout(width = 800, 
                  height = 600,
                  xaxis_title = "L1T Reduction Ratio",
                  yaxis_title = "L1T Skill Improvement",
                  title = "F1 Score by L1T Skill & Reduction Ratio",
                  legend=dict(xanchor = "right",
                    x = 0.95))
fig.update_xaxes(autorange="reversed")
fig.update_yaxes(range=[0.0, 0.8])
fig.show()

In [ ]:
fig = go.Figure(data =
    go.Contour(
        z=smoothed_productivity * 1e3,
        x=l1t_reductions, # horizontal axis
        y=l1t_skills, # vertical axis,
        contours = dict(showlabels = True),
        colorbar = dict(title = "Productivity (1/kJ)")
         
    ),
    )

y_offset = 0.015
fig.add_trace(go.Scatter(x = systems_reductions[0:1],
                        y = l1t_improvement[0:1] + y_offset,
                        mode = "markers",
                        marker = dict(size = 14, color = "green", symbol="circle"),
                        name = "Phase-1"))

fig.add_trace(go.Scatter(x = systems_reductions[1:2],
                        y = l1t_improvement[1:2],
                        mode = "markers",
                        marker = dict(size = 14, color = "green", symbol = "square"),
                        name = "L1 Tracks"))

fig.add_trace(go.Scatter(x = systems_reductions[2:3],
                        y = l1t_improvement[2:3] + y_offset,
                        mode = "markers",
                        marker = dict(size = 14, color = "green", symbol = "cross"),
                        name = "Increased L1T Accept"))

fig.add_trace(go.Scatter(x = systems_reductions[3:4],
                        y = l1t_improvement[3:4],
                        mode = "markers",
                        marker = dict(size = 14, color = "green", symbol = "star"),
                        name = "Phase-2"))

fig.update_layout(width = 800, 
                  height = 600,
                  xaxis_title = "L1T Reduction Ratio",
                  yaxis_title = "L1T Skill Improvement",
                  title = "Productivity by L1T Skill & Reduction Ratio",
                  legend=dict(xanchor = "right",
                    x = 0.95))
fig.update_xaxes(autorange="reversed")
fig.update_yaxes(range=[0.0, 0.8])
fig.add_annotation(x = -0.1, 
                   y = -0.1, 
                   showarrow=False,
                   text = "Pileup = 200", 
                   xref="paper", 
                   yref="paper",
                   font = dict(size = 14))
fig.show()

In [ ]:
import os
fig.write_image(os.path.join("figures", "systems contour.png"))

In [ ]:
fig = go.Figure(data =
    go.Contour(
        z=power,
        x=l1t_reductions, # horizontal axis
        y=l1t_skills, # vertical axis,
         contours_coloring='heatmap',
    ),
    )
fig.update_layout(width = 800, 
                  height = 600,
                  title = "Power by Trigger Skill & Reduction Ratio",
                  xaxis=dict(
                        title="Reduction Ratio",
                        titlefont=dict(size=24, family='Arial, bold')  # Bold font for the x-axis title
                    ),
                    yaxis=dict(
                        title="Skill",
                        titlefont=dict(size=24, family='Arial, bold')  # Bold font for the y-axis title
                    ),
                    font = dict(size=18,),)
fig.update_xaxes(autorange="reversed")
fig.show()

In [ ]:
#because its rejection is so much higher, there's more potential improvement gained by making L1T's skill higher 
#than simply passing more data to the HLT

In [ ]:
fig = go.Figure(data =
    go.Contour(
        z = smoothed_f1,
        x=power[0,:], # horizontal axis
        y=l1t_skills, # vertical axis,
         contours_coloring='heatmap',
         contours = dict(showlabels = True)
    ),
    )
fig.update_layout(width = 800,
                  height = 600,
                  title = "F1 Score by Skill Improvement and Power",
                  xaxis_title = "Power",
                  yaxis_title = "L1T Skill Improvement")
fig.update_yaxes(range=(0.0, 0.8))
fig.show()

In [ ]:
fig = go.Figure(data =
    go.Contour(
        z = np.transpose(smoothed_f1),
        y=power[0,:], # horizontal axis
        x=l1t_skills, # vertical axis,
         contours_coloring='heatmap',
         contours = dict(showlabels = True)
    ),
    )

fig.add_trace(go.Scatter(y = systems_power[0:1],
                        x = l1t_improvement[0:1],
                        mode = "markers",
                        marker = dict(size = 14, color = "gray"),
                        name = "Phase-1"))

fig.add_trace(go.Scatter(y = systems_power[1:2],
                        x = l1t_improvement[1:2],
                        mode = "markers",
                        marker = dict(size = 14, color = "red"),
                        name = "L1T Tracking"))

fig.add_trace(go.Scatter(y = systems_power[2:3],
                        x = l1t_improvement[2:3],
                        mode = "markers",
                        marker = dict(size = 14, color = "blue"),
                        name = "Increased L1T Accept"))

fig.add_trace(go.Scatter(y = systems_power[3:4],
                        x = l1t_improvement[3:4],
                        mode = "markers",
                        marker = dict(size = 14, color = "purple"),
                        name = "Phase-2"))

fig.add_trace(go.Scatter(y = systems_power[4:],
                        x = l1t_improvement[4:],
                        mode = "markers",
                        marker = dict(size = 14, color = "green"),
                        name = "Data Reduction"))

fig.update_layout(width = 800,
                  height = 600,
                  title = "F1 Score by Skill Improvement and Power",
                  yaxis_title = "Power",
                  xaxis_title = "L1T Skill Improvement",
                  legend=dict(xanchor = "right",
                    x = 0.95))
fig.update_xaxes(range=(0.0, 0.7))
fig.show()

In [ ]:
output_rate = np.array([1e3, 1e3, 7.5e3, 7.5e3, 7.5e3])

In [ ]:
output_rate

In [ ]:
# Execute ex_reduction to inspect node properties
ex_reduction_result = ex_reduction()

In [ ]:
productivity = (systems_f1 * output_rate) / systems_power

In [ ]:
systems_f1

In [ ]:
productivity

In [ ]:
fig = go.Figure(data =
    go.Bar(
        x = ["Phase-1", "L1T Tracking", "Increased L1T Accept", "Phase-2", "Data Reduction"],
        y= productivity
    ),
    )



fig.update_layout(width = 800,
                  height = 600,
                  title = "Productivity by System",
                  yaxis_title = "Productivity (Relevant Samples per Joule)",
                  xaxis_title = "System", )

fig.show()

In [ ]:
ex_reduction.get_node("Inner Tracker").parameters["sample data (B)"]

In [ ]:
ex_reduction.get_node("Intermediate").parameters["op efficiency (J/op)"]

In [ ]:
ex_reduction.get_node("Global").parameters["classifier (obj)"].skill_boost

In [ ]:
#final system model - tracker l1t upgrade w/ smart pixels

c0 = [ex_reduction.get_node("Inner Tracker").parameters["sample data (B)"],
      ex_reduction.get_node("HGCAL").parameters["sample data (B)"],
      53.3, #reduction ratio
      0.4, #l1t skill boost
      0.0, #hlt skill boost
      ex_reduction.get_node("Intermediate").parameters["op efficiency (J/op)"],
      ex_reduction.get_node("Global").parameters["op efficiency (J/op)"],]

In [ ]:
c0

In [ ]:
def vary_parameters(graph_def, tracker_data, hgcal_data, reduction_ratio, l1t_skill, hlt_skill, l1t_eff, hlt_eff):
    from systemflow.classifier import HLTClassifier
    l1t_cls = deepcopy(graph_def.get_node("Intermediate").parameters["classifier (obj)"])
    l1t_cls.skill_boost = l1t_skill
    hlt_cls = deepcopy(graph_def.get_node("Global").parameters["classifier (obj)"])
    hlt_cls.skill_boost = hlt_skill
    variant = hep_with_updated_parameters(graph_def, {
        "Inner Tracker": {"sample data (B)": tracker_data},
        "HGCAL": {"sample data (B)": hgcal_data},
        "Intermediate": {"reduction ratio (1)": reduction_ratio, "classifier (obj)": l1t_cls, "op efficiency (J/op)": l1t_eff},
        "Global": {"op efficiency (J/op)": hlt_eff, "classifier (obj)": hlt_cls}
    })
    result = variant()
    mv = result.metric_values
    power = mv["total power (W)"] / density_scale_model(2032)
    confusion = mv["pipeline contingency (2x2)"]
    prod = (f1_score(confusion) * 7500) / power
    return prod

In [ ]:
vary_parameters(ex_reduction, *c0)

In [ ]:
vp = lambda x: vary_parameters(ex_reduction, *x)

In [ ]:
c0a = np.array(c0)

In [ ]:
approx_fprime(c0a,
                vp,
                [1e2,
                 1e2,
                 1, 
                 0.05,
                 0.05,
                 1e-4,
                 1e-1],
              )